In [13]:
import numpy as np
import pandas as pd
from pathlib import Path

LAST_NAME = "PARRAS" 
STUDENT_ID = "TUPM-26-3597" 

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())

print("ID Sum:", ID_SUM)
print("Random Seed:", SEED_DIGIT)

WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"

assert measurement_path.is_file(), f"Missing file: {measurement_path}"
assert label_path.is_file(), f"Missing file: {label_path}"

ID Sum: 32
Random Seed: 7


In [14]:
measurements = np.genfromtxt(
    measurement_path,
    dtype=float,
    ndmin=2
)
labels = pd.read_csv(
    label_path,
    sep=r"\s+",
    header=None,
    names=["Outcome", "Timestamp"]
)

print("Measurement Shape:", measurements.shape)
print("Labels Shape:", labels.shape)

print("\nOutcome Values:")
print(labels["Outcome"].value_counts())

Measurement Shape: (1567, 590)
Labels Shape: (1567, 2)

Outcome Values:
Outcome
-1    1463
 1     104
Name: count, dtype: int64


In [15]:
#cell4
assert measurements.shape[0] == len(labels), \
    "Measurement and label record counts do not match."

feature_names = [
    f"Feature_{i + 1}" for i in range(measurements.shape[1])
]

#dataframe
df = pd.DataFrame(
    measurements,
    columns=feature_names
)
df.insert(0, "record_id", np.arange(1, len(df) + 1))
df["Outcome"] = labels["Outcome"].to_numpy()
df["Timestamp"] = labels["Timestamp"].to_numpy()
df["Outcome"] = df["Outcome"].map({
    -1: "Pass",
    1: "Fail"
})

print("Labeled DataFrame:")
display(df.head())

print("\nOutcome Counts:")
display(df["Outcome"].value_counts())

Labeled DataFrame:


,record_id,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Feature_6,Feature_7,Feature_8,Feature_9,...,Feature_583,Feature_584,Feature_585,Feature_586,Feature_587,Feature_588,Feature_589,Feature_590,Outcome,Timestamp
0,1,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,...,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,Pass,19/07/2008 11:55:00
1,2,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,...,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,Pass,19/07/2008 12:32:00
2,3,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,...,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,Fail,19/07/2008 13:17:00
3,4,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,...,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,Pass,19/07/2008 14:43:00
4,5,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,...,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,Pass,19/07/2008 15:22:00



Outcome Counts:


Outcome
Pass    1463
Fail     104
Name: count, dtype: int64

In [16]:
outcome_counts = df["Outcome"].value_counts()

pass_count = outcome_counts.get("Pass", 0)
fail_count = outcome_counts.get("Fail", 0)
sample_size = min(
    20 + ID_SUM,
    pass_count,
    fail_count
)

pass_sample = df[df["Outcome"] == "Pass"].sample(
    n=sample_size,
    random_state=SEED_DIGIT
)

fail_sample = df[df["Outcome"] == "Fail"].sample(
    n=sample_size,
    random_state=SEED_DIGIT
)

# Combine
balanced_sample = pd.concat(
    [pass_sample, fail_sample]
).sort_values("record_id").reset_index(drop=True)

print("Balanced Sample Size per Outcome:", sample_size)
print("\nSample Outcome Counts:")
display(balanced_sample["Outcome"].value_counts())

print("\nBalanced Sample Preview:")
display(balanced_sample[["record_id", "Outcome", "Timestamp"]].head())

Balanced Sample Size per Outcome: 52

Sample Outcome Counts:


Outcome
Fail    52
Pass    52
Name: count, dtype: int64


Balanced Sample Preview:


,record_id,Outcome,Timestamp
0,12,Fail,19/07/2008 22:52:00
1,39,Fail,28/07/2008 06:45:00
2,51,Fail,29/07/2008 18:08:00
3,58,Fail,30/07/2008 12:29:00
4,59,Fail,30/07/2008 21:16:00


In [17]:
measurement_df = df[feature_names].replace(
    [np.inf, -np.inf], np.nan
)

usable_features = []

for feature in feature_names:

    valid_values = measurement_df[feature].dropna()

    if len(valid_values) > 0 and valid_values.nunique() > 1:
        usable_features.append(feature)
assert len(usable_features) >= 5, "Fewer than five usable features found."


selected_features = usable_features[:5]
full_dataset_medians = measurement_df[selected_features].median()
df[selected_features] = measurement_df[selected_features].fillna(
    full_dataset_medians
)
balanced_sample = df.loc[
    df["record_id"].isin(balanced_sample["record_id"])
].copy()

print("Selected Features:", selected_features)
print("\nFull Dataset Medians:")
display(full_dataset_medians.to_frame(name="Median"))

print("\nMissing Values After Imputation:")
print(df[selected_features].isna().sum())

Selected Features: ['Feature_1', 'Feature_2', 'Feature_3', 'Feature_4', 'Feature_5']

Full Dataset Medians:


,Median
Feature_1,3011.4900
Feature_2,2499.4050
Feature_3,2201.0667
Feature_4,1285.2144
Feature_5,1.3168



Missing Values After Imputation:
Feature_1    0
Feature_2    0
Feature_3    0
Feature_4    0
Feature_5    0
dtype: int64


In [18]:
grouped = balanced_sample.groupby("Outcome")[selected_features]
group_stats = grouped.agg(
    ["count", "mean", "median", "std"]
)
print("GROUPED STATISTICS:")
display(group_stats)

GROUPED STATISTICS:


Feature_1                                   Feature_2               \
            count         mean    median        std     count         mean   
Outcome                                                                      
Fail           52  3003.780192  2990.490  81.924603        52  2495.413462   
Pass           52  3017.201346  3016.825  66.465085        52  2487.705385   

                             Feature_3                                      \
           median        std     count         mean      median        std   
Outcome                                                                      
Fail     2500.285  69.535176        52  2196.949142  2195.53330  30.463231   
Pass     2491.045  74.911719        52  2197.258335  2196.33335  21.536421   

        Feature_4                                      Feature_5            \
            count         mean      median         std     count      mean   
Outcome                                                                      
Fail           52  1364.885527  1274.48410  355.844489        52  1.275756   
Pass           52  1409.254519  1283.50805  514.770552        52  1.289029   

                            
          median       std  
Outcome                     
Fail     1.29835  0.309721  
Pass     1.32140  0.341831

In [19]:
pass_data = balanced_sample[
    balanced_sample["Outcome"] == "Pass"
]

fail_data = balanced_sample[
    balanced_sample["Outcome"] == "Fail"
]

pass_means = pass_data[selected_features].mean()
fail_means = fail_data[selected_features].mean()

absolute_mean_difference = (
    pass_means - fail_means
).abs()

pass_std = pass_data[selected_features].std(ddof=1)
fail_std = fail_data[selected_features].std(ddof=1)

# Get group sample sizes
n_pass = len(pass_data)
n_fail = len(fail_data)

pooled_std = np.sqrt(
    (
        (n_pass - 1) * pass_std**2
        + (n_fail - 1) * fail_std**2
    ) / (n_pass + n_fail - 2)
)

standardized_mean_difference = pd.Series(
    np.divide(
        absolute_mean_difference.to_numpy(),
        pooled_std.to_numpy(),
        out=np.where(
            absolute_mean_difference.to_numpy() == 0,
            0.0,
            np.inf
        ),
        where=pooled_std.to_numpy() != 0
    ),
    index=selected_features
)

print("ABSOLUTE MEAN DIFFERENCES:")
display(absolute_mean_difference.to_frame(name="Absolute Mean Difference"))

print("STANDARDIZED MEAN DIFFERENCES:")
display(standardized_mean_difference.to_frame(name="Absolute SMD"))

ABSOLUTE MEAN DIFFERENCES:


,Absolute Mean Difference
Feature_1,13.421154
Feature_2,7.708077
Feature_3,0.309192
Feature_4,44.368992
Feature_5,0.013273


STANDARDIZED MEAN DIFFERENCES:


,Absolute SMD
Feature_1,0.179917
Feature_2,0.106652
Feature_3,0.011721
Feature_4,0.100269
Feature_5,0.040694


In [20]:
feature_comparison = pd.DataFrame({
    "Feature": selected_features,
    "Pass_Mean": pass_means.values,
    "Fail_Mean": fail_means.values,
    "Pass_Std": pass_std.values,
    "Fail_Std": fail_std.values,
    "Pooled_Std": pooled_std.values,
    "Absolute_Mean_Difference": absolute_mean_difference.values,
    "Absolute_Standardized_Mean_Difference":
        standardized_mean_difference.values
})

feature_comparison = feature_comparison.sort_values(
    by="Absolute_Standardized_Mean_Difference",
    ascending=False
).reset_index(drop=True)

feature_comparison.insert(
    0,
    "Rank",
    range(1, len(feature_comparison) + 1)
)

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

balanced_sample.to_csv(
    output_dir / "balanced_sample.csv",
    index=False
)
feature_comparison.to_csv(
    output_dir / "feature_comparison.csv",
    index=False
)

print("FEATURE COMPARISON:")
display(feature_comparison)

print("\nFiles successfully exported:")
print((output_dir / "balanced_sample.csv").resolve())
print((output_dir / "feature_comparison.csv").resolve())

FEATURE COMPARISON:


,Rank,Feature,Pass_Mean,Fail_Mean,Pass_Std,Fail_Std,Pooled_Std,Absolute_Mean_Difference,Absolute_Standardized_Mean_Difference
0,1,Feature_1,3017.201346,3003.780192,66.465085,81.924603,74.596408,13.421154,0.179917
1,2,Feature_2,2487.705385,2495.413462,74.911719,69.535176,72.273461,7.708077,0.106652
2,3,Feature_4,1409.254519,1364.885527,514.770552,355.844489,442.500860,44.368992,0.100269
3,4,Feature_5,1.289029,1.275756,0.341831,0.309721,0.326172,0.013273,0.040694
4,5,Feature_3,2197.258335,2196.949142,21.536421,30.463231,26.380161,0.309192,0.011721



Files successfully exported:
C:\Users\JULIANNA\OneDrive\Desktop\VISUAL CODE\Lab7_NumPy_Pandas_Parras\output\balanced_sample.csv
C:\Users\JULIANNA\OneDrive\Desktop\VISUAL CODE\Lab7_NumPy_Pandas_Parras\output\feature_comparison.csv


In [21]:
numpy_pandas_match = True

for outcome in ["Pass", "Fail"]:

    group_data = balanced_sample.loc[
        balanced_sample["Outcome"] == outcome,
        selected_features
    ].to_numpy(dtype=float)

    numpy_count = np.sum(~np.isnan(group_data), axis=0)
    numpy_mean = np.nanmean(group_data, axis=0)
    numpy_median = np.nanmedian(group_data, axis=0)
    numpy_std = np.nanstd(group_data, axis=0, ddof=1)
    pandas_stats = group_stats.loc[outcome]

    pandas_count = pandas_stats.xs("count", level=1).to_numpy()
    pandas_mean = pandas_stats.xs("mean", level=1).to_numpy()
    pandas_median = pandas_stats.xs("median", level=1).to_numpy()
    pandas_std = pandas_stats.xs("std", level=1).to_numpy()

    checks = [
        np.array_equal(numpy_count, pandas_count),
        np.allclose(numpy_mean, pandas_mean, equal_nan=True),
        np.allclose(numpy_median, pandas_median, equal_nan=True),
        np.allclose(numpy_std, pandas_std, equal_nan=True)
    ]

    if not all(checks):
        numpy_pandas_match = False

print(
    "NumPy-Pandas Statistics Match:",
    "YES" if numpy_pandas_match else "NO"
)

NumPy-Pandas Statistics Match: YES


In [24]:
#cell11
highest_ranked_feature = feature_comparison.iloc[0]["Feature"]
highest_absolute_smd = feature_comparison.iloc[0][
    "Absolute_Standardized_Mean_Difference"
]

print("Balanced Sample Size:")
print(f"{sample_size} records per outcome")
print(f"{len(balanced_sample)} records total")
print("Selected Features:", selected_features)
print("Highest-Ranked Feature:", highest_ranked_feature)
print("Absolute Standardized Mean Difference:", f"{highest_absolute_smd:.6f}")
print("NumPy-Pandas Statistics Match:", "YES" if numpy_pandas_match else "NO")



Balanced Sample Size:
52 records per outcome
104 records total
Selected Features: ['Feature_1', 'Feature_2', 'Feature_3', 'Feature_4', 'Feature_5']
Highest-Ranked Feature: Feature_1
Absolute Standardized Mean Difference: 0.179917
NumPy-Pandas Statistics Match: YES
